# PBI Package Tutorial

This comprehensive tutorial demonstrates how to use the PBI (Phage-Bacteria Interaction) Python package for analyzing phage-host interaction data.## What You'll Learn1. **Database Connection**: Connect to the PBI database and explore its contents2. **Data Retrieval**: Query and filter phage-host interaction data3. **Sequence Access**: Retrieve phage and host genome sequences4. **Log Analysis**: Understand tracking and diagnostic information5. **Streaming Datasets**: Use memory-efficient datasets for machine learning6. **Architecture**: Understand how the components work together## Architecture Overview

```
┌─────────────────────────────────────────────────────────┐
│                    PBI Architecture
│
├─────────────────────────────────────────────────────────┤
│
│
│  ┌──────────────┐     ┌─────────────┐                   │
│  │  DuckDB      │───▶│ Metadata    │                   │
│  │  Database    │     │ Queries     │                   │
│  └──────────────┘     └─────────────┘                   │
│         │                                               │
│         │                                               │
│  ┌──────▼──────────────────────────────────────┐        │
│  │     SequenceRetriever                       │        │
│  │  - Lazy FASTA loading                       │        │
│  │  - Index-based sequence access              │        │
│  │  - Query + sequence integration             │        │ 
│  └──────┬──────────────────────────────────────┘        │
│         │                                               │
│         ├─────────────┬──────────────┬─────────────┐    │
│         │             │              │             │    │
│  ┌──────▼──────┐ ┌───▼────┐  ┌──────▼──────┐ ┌───▼──┐   │
│  │Indexed FASTA│ │Host    │  │Streaming    │ │Index-│   │
│  │- Phages     │ │Mapping │  │Dataset      │ │ed    │   │
│  │- Proteins   │ │JSON    │  │(ML)         │ │Data  │   │
│  └─────────────┘ └────────┘  └─────────────┘ └──────┘   │
│                                                         │
└─────────────────────────────────────────────────────────┘
```
**Key Components:**
- **DuckDB Database**: Stores metadata (taxonomy, assembly info, etc.)
- **Indexed FASTA Files**: Store sequences with `.fai` index for fast random access
- **SequenceRetriever**: Main interface combining metadata queries with sequence retrieval
- **Host Mapping JSON**: Maps Host_ID to individual genome FASTA files
- **Datasets**: PyTorch-compatible datasets for machine learning workflows

---## 1. Installation and SetupFirst, let's ensure the PBI package is properly installed and import necessary libraries.

In [ ]:
# Import PBI and other useful librariesimport pbiimport pandas as pdimport timefrom pathlib import Path# Set pandas display options for better readabilitypd.set_option('display.max_columns', None)pd.set_option('display.width', None)pd.set_option('display.max_colwidth', 50)print("✅ PBI package imported successfully")print(f"   Version: {pbi.__version__ if hasattr(pbi, '__version__') else 'dev'}")

---## 2. Database Connection and ExplorationThe PBI package provides a convenient `quick_connect()` function that automatically finds and connects to your database.### How Connection Works:1. **Auto-detection**: Searches common locations for database and FASTA files2. **Lazy Loading**: Database connects immediately, but FASTA files load in background3. **Ready to Query**: Can query database while sequences are being indexedThis design allows you to start working immediately without waiting for large FASTA files to load.

In [ ]:
# Quick connect (auto-detects database and FASTA files)print("🔍 Connecting to PBI database...")start = time.time()retriever = pbi.quick_connect()elapsed = time.time() - startprint(f"✅ Connected in {elapsed:.2f} seconds")print(f"   Database: {retriever.conn}")print(f"   Phage FASTA: {'Ready' if retriever._phage_fasta else 'Loading...'}")print(f"   Host data: {'Available' if retriever._has_host_data else 'Not loaded'}")

### Database StatisticsLet's explore what's in the database:

In [ ]:
# Get comprehensive statisticsstats = retriever.get_stats()print("📊 Database Statistics:")print("=" * 60)print(f"Phages:   {stats['database']['phages']:,}")print(f"Proteins: {stats['database']['proteins']:,}")print(f"Hosts:    {stats['database']['hosts']:,}")print(f"Phage-Host Associations: {stats['database']['phage_host_associations']:,}")print()print("FASTA Files:")print(f"  Phage sequences:   {stats['fasta']['phage_sequences']:,}")print(f"  Protein sequences: {stats['fasta']['protein_sequences']:,}")

---## 3. Data Retrieval and FilteringNow let's query the database to retrieve specific data. We'll explore different filtering options.### Understanding the Data ModelThe database uses a **star schema** with fact and dimension tables:- **fact_phages**: Main phage information (length, GC content, taxonomy)- **dim_hosts**: Host bacterial information (species, assembly, genome stats)- **dim_proteins**: Protein sequences and annotations- **phage_host_associations**: Links phages to their hostsYou can filter on any column using SQL WHERE clause syntax.

In [ ]:
# Example 1: Get complete phages with high GC contentprint("Example 1: Complete phages with GC > 60%")print("-" * 60)phage_metadata = retriever.get_phage_metadata(    where_clause="p.Completeness = 'complete' AND p.GC_content > 60",    limit=10)print(f"Found {len(phage_metadata)} phages")print(phage_metadata[['Phage_ID', 'Length', 'GC_content', 'Taxonomy']].head())

In [ ]:
# Example 2: Get hosts with complete genomesprint("\nExample 2: Hosts with complete genome assemblies")print("-" * 60)host_metadata = retriever.get_host_metadata(    where_clause="h.Assembly_Level = 'Complete Genome'",    limit=10)print(f"Found {len(host_metadata)} hosts")print(host_metadata[['Species_Name', 'Assembly_Level', 'Genome_Length', 'GC_Content']].head())

In [ ]:
# Example 3: Complex filter - Phage-host pairs with specific criteriaprint("\nExample 3: Large phages (>100kb) infecting Escherichia")print("-" * 60)phage_host_data = retriever.get_phage_host_metadata(    where_clause="p.Length > 100000 AND h.Species_Name LIKE 'Escherichia%'",    limit=10)print(f"Found {len(phage_host_data)} interactions")print(phage_host_data[['Phage_ID', 'Phage_Length', 'Host_Species', 'Host_Assembly_Level']].head())

### Using LIMIT and OFFSETThe `where_clause` parameter now supports `LIMIT` and `OFFSET` directly:

In [ ]:
# Example 4: Pagination with LIMIT and OFFSETprint("Example 4: Pagination - Get 2nd batch of 5 phages")print("-" * 60)# First 5 phagesbatch1 = retriever.get_phage_metadata(where_clause="LIMIT 5")print(f"Batch 1: {len(batch1)} phages")print(batch1['Phage_ID'].tolist())# Next 5 phages (using OFFSET)batch2 = retriever.get_phage_metadata(where_clause="LIMIT 5 OFFSET 5")print(f"\nBatch 2: {len(batch2)} phages")print(batch2['Phage_ID'].tolist())# Combined filter + limitbatch3 = retriever.get_phage_metadata(    where_clause="p.Completeness = 'complete' LIMIT 5")print(f"\nBatch 3 (filtered): {len(batch3)} complete phages")print(batch3['Phage_ID'].tolist())

---## 4. Sequence Retrieval (Non-Streaming)Now let's retrieve actual DNA sequences. The `get_phage_host_pairs()` method combines metadata with sequences.### How Sequence Retrieval Works:1. **Query Database**: Get metadata matching your filter2. **Fetch from FASTA**: Use `.fai` index to quickly locate sequences3. **Combine**: Return DataFrame with both metadata and sequencesThis is **non-streaming** - all data is loaded into memory. Good for small to medium datasets (<10,000 pairs).

In [ ]:
# Example 5: Get phage-host pairs with sequencesprint("Example 5: Retrieve phage-host pairs with sequences")print("-" * 60)# Get 5 pairspairs = retriever.get_phage_host_pairs(    where_clause="LIMIT 5")print(f"Retrieved {len(pairs)} phage-host pairs with sequences")print("\nMetadata columns:", list(pairs.columns))print("\nSample data:")print(pairs[['Phage_ID', 'Species_Name', 'Phage_Length', 'Host_Length']].head())# Check sequence contentprint("\nSequence preview:")for idx, row in pairs.head(2).iterrows():    print(f"  Phage {row['Phage_ID'][:30]}: {row['Phage_Sequence'][:50]}...")    print(f"  Host {row['Species_Name'][:30]}: {row['Host_Sequence'][:50]}...")    print()

In [ ]:
# Example 6: Filter by taxonomyprint("Example 6: Get Siphoviridae phages and their hosts")print("-" * 60)sipho_pairs = retriever.get_phage_host_pairs(    where_clause="p.Taxonomy LIKE '%Siphoviridae%' LIMIT 10")print(f"Found {len(sipho_pairs)} Siphoviridae phage-host pairs")if len(sipho_pairs) > 0:    print("\nTaxonomy distribution:")    print(sipho_pairs['Phage_Taxonomy'].value_counts())

---## 5. Log Analysis and Tracking InformationUnderstanding where to find diagnostic information is crucial for troubleshooting and quality control.### Types of Tracking Information:1. **Host Download Metadata** (`host_metadata.csv`) - Successful downloads2. **Failure Logs** (`failed_downloads.txt`) - What couldn't be downloaded3. **Missing Hosts CSV** (from datasets) - Runtime missing data tracking4. **Snakemake Logs** - Pipeline execution logsLet's examine each:

In [ ]:
# Example 7: Analyze host download metadataprint("Example 7: Host Download Metadata Analysis")print("-" * 60)# Path to metadata (adjust based on your setup)metadata_path = Path("data/processed/host_metadata.csv")if metadata_path.exists():    host_meta = pd.read_csv(metadata_path)        print(f"Total host genomes downloaded: {len(host_meta)}")    print(f"\nAssembly level distribution:")    print(host_meta['Assembly_Level'].value_counts())        print(f"\nSequence count statistics:")    print(f"  Mean: {host_meta['Sequence_Count'].mean():.1f}")    print(f"  Median: {host_meta['Sequence_Count'].median():.0f}")    print(f"  Max: {host_meta['Sequence_Count'].max():.0f}")        # Identify highly fragmented genomes    fragmented = host_meta[host_meta['Sequence_Count'] > 100]    print(f"\nHighly fragmented genomes (>100 sequences): {len(fragmented)}")    if len(fragmented) > 0:        print(fragmented[['Species_Name', 'Assembly_Level', 'Sequence_Count']].head())        # GC content distribution    print(f"\nGC content range: {host_meta['GC_Content'].min():.1f}% - {host_meta['GC_Content'].max():.1f}%")    else:    print(f"⚠️  Metadata file not found at: {metadata_path}")    print("   Run the host genome download pipeline first.")    print("   See: docs/guides/pipeline-execution.md")

In [ ]:
# Example 8: Check for common issuesprint("\nExample 8: Quality Control Checks")print("-" * 60)if metadata_path.exists():    host_meta = pd.read_csv(metadata_path)        # Check 1: Very small genomes (possible issues)    small = host_meta[host_meta['Genome_Length'] < 1_000_000]    print(f"Small genomes (<1 Mbp): {len(small)}")    if len(small) > 0:        print("  These may be plasmids or incomplete downloads")        print(small[['Species_Name', 'Genome_Length', 'Assembly_Level']].head())        # Check 2: Extreme GC content (possible contamination)    extreme_gc = host_meta[(host_meta['GC_Content'] < 30) | (host_meta['GC_Content'] > 75)]    print(f"\nExtreme GC content (<30% or >75%): {len(extreme_gc)}")    if len(extreme_gc) > 0:        print(extreme_gc[['Species_Name', 'GC_Content', 'Assembly_Level']].head())        # Check 3: Recent downloads    host_meta['Download_Date'] = pd.to_datetime(host_meta['Download_Date'])    recent = host_meta[host_meta['Download_Date'] > pd.Timestamp.now() - pd.Timedelta(days=30)]    print(f"\nGenomes downloaded in last 30 days: {len(recent)}")else:    print("Run Example 7 first to load metadata")

### Reading Failure LogsWhen genomes fail to download, understanding why helps improve data completeness:

In [ ]:
# Example 9: Analyze failure logprint("Example 9: Download Failure Analysis")print("-" * 60)failure_log = Path("data/logs/failed_downloads.txt")if failure_log.exists():    with open(failure_log) as f:        failures = f.read()        # Count failures by category    import re    categories = re.findall(r'Category: (.*?) \((\d+)', failures)        print("Failure breakdown:")    for category, count in categories:        print(f"  {category}: {count}")        # Show sample failures    print("\nSample failed species:")    species_matches = re.findall(r'^  - (.+)$', failures, re.MULTILINE)    for species in species_matches[:5]:        print(f"  • {species}")else:    print(f"⚠️  Failure log not found at: {failure_log}")    print("   This is normal if all downloads succeeded!")

### Runtime Missing Hosts TrackingWhen using datasets, you can track which phage-host pairs have missing hosts:

In [ ]:
# Example 10: Track missing hosts during dataset usageprint("Example 10: Runtime Missing Hosts Tracking")print("-" * 60)# Create a dataset with missing hosts trackingmissing_csv = Path("/tmp/missing_hosts_example.csv")try:    dataset = retriever.create_indexed_dataset(        where_clause="LIMIT 100",        missing_hosts_csv=str(missing_csv)    )        print(f"Created dataset with {len(dataset)} samples")        # Iterate through a few samples (this triggers host lookup)    for i, sample in enumerate(dataset):        if i >= 10:  # Just check first 10            break        # Check if any hosts were missing    if missing_csv.exists():        missing = pd.read_csv(missing_csv)        print(f"\nMissing hosts found: {len(missing)}")        if len(missing) > 0:            print("\nFailure reasons:")            print(missing['Failure_Reason'].value_counts())            print("\nSample missing hosts:")            print(missing[['Phage_ID', 'Species_Name', 'Failure_Reason']].head())    else:        print("✅ All hosts were successfully retrieved!")        except Exception as e:    print(f"⚠️  Could not create dataset: {e}")    print("   Make sure host genomes are downloaded and indexed.")

---## 6. Streaming Datasets for Machine LearningFor large-scale machine learning, loading all data into memory isn't feasible. **Streaming datasets** solve this by loading data on-demand.### Streaming vs. Indexed Datasets:| Feature | Indexed Dataset | Streaming Dataset ||---------|----------------|-------------------|| Memory Usage | High (all metadata in RAM) | Low (batches only) || Random Access | Yes (shuffling supported) | No (sequential) || Use Case | Medium datasets (<100k pairs) | Large datasets (>100k pairs) || PyTorch DataLoader | ✅ Full support | ✅ Full support |### How Streaming Works:```Database Query (batched)    ↓Fetch batch of metadata    ↓Load sequences for batch    ↓Yield samples one by one    ↓Repeat until exhausted```This keeps memory constant regardless of dataset size.

In [ ]:
# Example 11: Create and use streaming datasetprint("Example 11: Streaming Dataset")print("-" * 60)# Create streaming datasetstream_dataset = retriever.create_streaming_dataset(    where_clause="LIMIT 50",  # Small for demo    batch_size=10  # Fetch 10 at a time from DB)print(f"Created streaming dataset")print(f"  Batch size: 10")print(f"  Total (estimated): ~50 samples")# Iterate through datasetprint("\nStreaming samples:")for i, sample in enumerate(stream_dataset):    if i >= 5:  # Show first 5        break    print(f"  Sample {i+1}:")    print(f"    Phage: {sample['Phage_ID'][:40]}")    print(f"    Host: {sample['Species_Name'][:40]}")    print(f"    Phage length: {sample['Phage_Length']:,} bp")    print(f"    Has sequence: {len(sample['Phage_Sequence']) > 0}")    print()print("✅ Streaming completed")

In [ ]:
# Example 12: Using datasets with PyTorch (if available)print("Example 12: PyTorch Integration")print("-" * 60)try:    from torch.utils.data import DataLoader    from pbi import phage_host_collate_fn        # Create indexed dataset (supports shuffling)    dataset = retriever.create_indexed_dataset(        where_clause="LIMIT 100"    )        # Create DataLoader with custom collate function    dataloader = DataLoader(        dataset,        batch_size=8,        shuffle=True,  # Randomly shuffle samples        num_workers=0,  # Set to 2-4 for parallel loading        collate_fn=phage_host_collate_fn  # Handle mixed types    )        print(f"Created DataLoader:")    print(f"  Dataset size: {len(dataset)}")    print(f"  Batch size: 8")    print(f"  Num batches: {len(dataloader)}")        # Get one batch    batch = next(iter(dataloader))    print(f"\nBatch structure:")    print(f"  Type: {type(batch)}")    print(f"  Keys: {list(batch.keys())}")    print(f"  Phage_IDs in batch: {len(batch['Phage_ID'])}")    print(f"  Sample Phage_ID: {batch['Phage_ID'][0][:40]}")    except ImportError:    print("⚠️  PyTorch not installed")    print("   Install with: pip install torch")    print("   Datasets will still work, but without DataLoader features")

### Context Manager for CleanupBoth dataset types support Python's context manager protocol for automatic cleanup:

In [ ]:
# Example 13: Using context manager (recommended)print("Example 13: Context Manager (Automatic Cleanup)")print("-" * 60)# Using 'with' ensures files are closed properlywith retriever.create_indexed_dataset(where_clause="LIMIT 20") as dataset:    print(f"Dataset opened: {len(dataset)} samples")        # Use dataset    sample = dataset[0]    print(f"First sample: {sample['Phage_ID'][:40]}")        # Files automatically closed when exiting 'with' blockprint("✅ Dataset closed automatically")# Alternative: Manual cleanupdataset = retriever.create_indexed_dataset(where_clause="LIMIT 20")try:    print(f"\nDataset opened: {len(dataset)} samples")finally:    dataset.close()  # Explicit cleanup    print("✅ Dataset closed manually")

---## 7. Custom TransformationsYou can apply custom transformations to samples for preprocessing:

In [ ]:
# Example 14: Custom transformationprint("Example 14: Custom Transformation")print("-" * 60)def sequence_encoding_transform(sample):    """Example: Convert sequences to one-hot encoding (simplified)"""    # Add custom fields    sample['Phage_Length_Category'] = (        'Large' if sample['Phage_Length'] > 50000 else 'Small'    )        # You could add:    # - One-hot encoding of sequences    # - K-mer features    # - GC content windows    # - Etc.        return sample# Create dataset with transformtransformed_dataset = retriever.create_indexed_dataset(    where_clause="LIMIT 10",    transform=sequence_encoding_transform)# Check transformed samplesample = transformed_dataset[0]print("Transformed sample keys:")print(list(sample.keys()))print(f"\nNew field - Phage_Length_Category: {sample['Phage_Length_Category']}")print(f"Original Phage_Length: {sample['Phage_Length']:,} bp")

---## 8. Summary and Best Practices### What We Covered✅ **Database Connection**: Quick setup with auto-detection  ✅ **Data Filtering**: Flexible SQL-based queries  ✅ **Sequence Retrieval**: Both non-streaming and streaming approaches  ✅ **Log Analysis**: Understanding downloads and failures  ✅ **ML Integration**: PyTorch-compatible datasets  ✅ **Resource Management**: Context managers and cleanup  ### Best Practices#### For Data Exploration (Notebooks)- Use `get_phage_host_pairs()` for small queries (<10k pairs)- Use `LIMIT` to preview data before full queries- Check metadata CSVs before working with sequences#### For Machine Learning- Use **IndexedDataset** for medium datasets with shuffling- Use **StreamingDataset** for large datasets (>100k pairs)- Always use `with` context manager or call `close()`- Use `missing_hosts_csv` to track data quality#### For Production- Monitor failure logs during downloads- Validate `Sequence_Count` for quality control- Archive logs with dates for reproducibility- Use NCBI API key for faster downloads### Next Steps📖 **Documentation:**- [Pipeline Execution Guide](../docs/guides/pipeline-execution.md) - Run the full pipeline- [Analysis Guide](../docs/guides/analysis-guide.md) - Advanced queries- [Machine Learning Guide](../docs/guides/machine-learning.md) - Training models🔬 **Example Notebooks:**- `example_streaming_ml.ipynb` - Full ML workflow- `ml_1_phage_host_dataset.ipynb` - Dataset details- `analysis_direct_access_guide.ipynb` - Database queries### Getting Help```python# Built-in helpretriever.help()# Package documentationhelp(pbi.SequenceRetriever)```Happy analyzing! 🦠🔬